# BiomedBERT + ResNet 1D + CRF cho DrugNER

Kiến trúc: **BiomedBERT → projection → 3 residual Conv1D blocks → Linear → CRF**.
Mỗi block gồm hai Conv1D (kernel 3, stride 1), LayerNorm theo từng vị trí,
ReLU, dropout và kết nối tắt. Mặc định dilation = 1; không pooling.
CRF dùng negative log-likelihood và Viterbi với ràng buộc BIO.

Notebook độc lập, không cần upload `resnet_crf.py`, không yêu cầu W&B/Kaggle Secrets.
Cần DDICorpus và Internet ở lần đầu tải tokenizer/encoder từ Hugging Face.
Nếu môi trường thiếu thư viện, chạy `%pip install "torch>=2.3,<3" "transformers>=4.40,<5" numpy pandas tqdm`.

**Quy ước đánh giá:** giữ cách chia train/validation theo câu và nhãn BIO theo
subword của `BERT_CRF.ipynb` để thuận tiện đối chiếu. `[CLS]`, `[SEP]` và padding
bị loại khỏi loss/giải mã CRF. Báo cáo micro-F1 khớp loại và ranh giới thực thể
trên chuỗi token, không phải official DDI character-offset scorer.
Các thực thể không liên tục được tách thành từng đoạn như baseline; flat BIO không
biểu diễn được thực thể chồng lấn. Câu vượt MAX_LENGTH bị cắt; cần tăng độ dài hoặc
triển khai windowing khi đánh giá đầy đủ văn bản dài. Muốn đánh giá tổng quát hóa
sang tài liệu mới, dùng cùng một phép chia theo document cho tất cả baseline.

Mô hình này là bản triển khai cho dự án, không phải mã nguồn gốc hay bản tái lập
baseline ResNet-CRF trong paper Bhushan et al.


In [2]:
import os
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from transformers.utils import import_utils as transformers_import_utils



In [3]:
# Reproducibility
SEED = 23022006
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA devices:", torch.cuda.device_count())

Device: cuda
GPU: Tesla T4
CUDA devices: 2


In [4]:
# Config
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

MAX_LENGTH = 192
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
EPOCHS = 10
VALID_RATIO = 0.1

LR_BERT = 2e-5
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
BEST_MODEL_PATH = OUTPUT_DIR / "biomedbert_resnet_crf_ddi_best.pt"
LABEL_PATH = OUTPUT_DIR / "ddi_resnet_crf_label_mapping.json"
RESNET_CHANNELS = 256
RESNET_BLOCKS = 3
RESNET_KERNEL_SIZE = 3
RESNET_DILATIONS = [1, 1, 1]
DROPOUT = 0.2
NUM_WORKERS = 0  # Portable in local notebooks and Kaggle.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CONFIG = {
    "channels": RESNET_CHANNELS,
    "num_blocks": RESNET_BLOCKS,
    "kernel_size": RESNET_KERNEL_SIZE,
    "dilations": RESNET_DILATIONS,
    "dropout": DROPOUT,
}


In [5]:
def find_ddi_root():
    configured = os.environ.get("DDI_ROOT")
    if configured:
        path = Path(configured).expanduser()
        if not path.is_dir():
            raise FileNotFoundError(f"DDI_ROOT does not exist: {path}")
        return path
    candidates = [
        Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus"),
        Path("/kaggle/input/dataset/DDICorpus"),
        Path("/kaggle/input/datasets/DDICorpus"),
        Path("/kaggle/input/DDICorpus"),
        Path("dataset/DDICorpus"),
        Path("DDICorpus"),
    ]
    for path in candidates:
        if path.exists():
            return path

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob("DDICorpus"))
        if matches:
            return matches[0]

    raise FileNotFoundError("Khong tim thay folder DDICorpus. Hay kiem tra lai Kaggle input path.")

DDI_ROOT = find_ddi_root()
TRAIN_DIR = DDI_ROOT / "Train"
TEST_ROOT = DDI_ROOT / "Test"
TEST_DRUG_NER_DIR = TEST_ROOT / "Test for DrugNER task"
# Never mix DrugNER annotations with the separate DDI relation-extraction test set.
# If using a different corpus layout, point TEST_DIR explicitly to its NER split.
TEST_DIR = TEST_DRUG_NER_DIR

print("DDI_ROOT:", DDI_ROOT)
print("TRAIN_DIR:", TRAIN_DIR, "| exists:", TRAIN_DIR.exists())
print("TEST_DIR:", TEST_DIR, "| exists:", TEST_DIR.exists())
print("Train XML files:", len(list(TRAIN_DIR.rglob("*.xml"))))
print("Test XML files:", len(list(TEST_DIR.rglob("*.xml"))) if TEST_DIR.exists() else 0)

DDI_ROOT: /kaggle/input/datasets/tuantc2306/dataset/DDICorpus
TRAIN_DIR: /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train | exists: True
TEST_DIR: /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Test/Test for DrugNER task | exists: True
Train XML files: 714
Test XML files: 112


In [6]:
def parse_char_offsets(offset_text):
    """DDI offsets are inclusive, sometimes discontinuous: '0-5;10-14'."""
    spans = []
    if not offset_text:
        return spans
    for part in offset_text.split(";"):
        part = part.strip()
        if not part or "-" not in part:
            continue
        start, end = part.split("-", 1)
        try:
            start, end = int(start), int(end)
        except ValueError:
            continue
        if end >= start:
            spans.append((start, end + 1))
    return spans


def read_xml_file(path):
    try:
        root = ET.parse(path).getroot()
    except ET.ParseError:
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
        root = ET.fromstring(text)

    examples = []
    for sent in root.iter("sentence"):
        text = sent.attrib.get("text", "")
        if not text:
            continue

        entities = []
        for ent in sent.findall("entity"):
            ent_type = ent.attrib.get("type", "drug")
            ent_text = ent.attrib.get("text", "")
            char_offset = ent.attrib.get("charOffset", "")
            for start, end in parse_char_offsets(char_offset):
                if 0 <= start < end <= len(text):
                    entities.append({
                        "start": start,
                        "end": end,
                        "type": ent_type,
                        "text": ent_text,
                    })

        entities = sorted(entities, key=lambda x: (x["start"], x["end"]))
        examples.append({"id": sent.attrib.get("id", ""), "text": text, "entities": entities})
    return examples


def load_ddi_examples(folder):
    xml_files = sorted(Path(folder).rglob("*.xml"))
    examples = []
    skipped = []
    for xml_file in tqdm(xml_files, desc=f"Reading {folder}"):
        try:
            examples.extend(read_xml_file(xml_file))
        except Exception as exc:
            skipped.append((str(xml_file), str(exc)))
    return examples, skipped

train_examples_all, skipped_train = load_ddi_examples(TRAIN_DIR)
test_examples, skipped_test = load_ddi_examples(TEST_DIR) if TEST_DIR.exists() else ([], [])

print("Train sentences:", len(train_examples_all), "| skipped XML:", len(skipped_train))
print("Test sentences:", len(test_examples), "| skipped XML:", len(skipped_test))
print("Example:", train_examples_all[0] if train_examples_all else None)
if not train_examples_all:
    raise ValueError("No training sentences loaded; check TRAIN_DIR and XML files.")
if skipped_train or skipped_test:
    raise ValueError(f"XML parsing failed: {(skipped_train + skipped_test)[:5]}")
if not test_examples:
    warnings.warn("No DrugNER test data loaded. Set TEST_DIR to the NER test split.")


Reading /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train:   0%|          | 0/714 [00:00<?, ?it/s]

Reading /kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Test/Test for DrugNER task:   0%|          | 0/112…

Train sentences: 6905 | skipped XML: 0
Test sentences: 665 | skipped XML: 0
Example: {'id': 'DDI-DrugBank.d436.s0', 'text': 'No drug, nutritional supplement, food or herb interactions have yet been reported.', 'entities': []}


In [7]:
def entity_type_counts(examples):
    counter = Counter()
    sent_with_entity = 0
    for ex in examples:
        if ex["entities"]:
            sent_with_entity += 1
        for ent in ex["entities"]:
            counter[ent["type"]] += 1
    return counter, sent_with_entity

train_type_counts, train_sent_with_ent = entity_type_counts(train_examples_all)
test_type_counts, test_sent_with_ent = entity_type_counts(test_examples)

print("Train entity types:", train_type_counts)
print("Train sentences with entity:", train_sent_with_ent)
print("Test entity types:", test_type_counts)
print("Test sentences with entity:", test_sent_with_ent)

Train entity types: Counter({'drug': 9432, 'group': 3429, 'brand': 1437, 'drug_n': 505})
Train sentences with entity: 5560
Test entity types: Counter({'drug': 352, 'group': 156, 'drug_n': 121, 'brand': 59})
Test sentences with entity: 324


In [8]:
# Split train into train/validation by sentence.
rng = random.Random(SEED)
indices = list(range(len(train_examples_all)))
rng.shuffle(indices)
if len(indices) < 2 or not 0 < VALID_RATIO < 1:
    raise ValueError("Need at least two sentences and 0 < VALID_RATIO < 1.")
valid_size = min(len(indices) - 1, max(1, int(len(indices) * VALID_RATIO)))
valid_idx = set(indices[:valid_size])

train_examples = [ex for i, ex in enumerate(train_examples_all) if i not in valid_idx]
valid_examples = [ex for i, ex in enumerate(train_examples_all) if i in valid_idx]

entity_types = sorted({ent["type"] for ex in train_examples_all for ent in ex["entities"]})
labels = ["O"]
for ent_type in entity_types:
    labels.extend([f"B-{ent_type}", f"I-{ent_type}"])

label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Train:", len(train_examples), "| Valid:", len(valid_examples), "| Test:", len(test_examples))
print("Labels:", labels)

with open(LABEL_PATH, "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label, "model_name": MODEL_NAME}, f, indent=2)
print("Saved label mapping to", LABEL_PATH)

Train: 6215 | Valid: 690 | Test: 665
Labels: ['O', 'B-brand', 'I-brand', 'B-drug', 'I-drug', 'B-drug_n', 'I-drug_n', 'B-group', 'I-group']
Saved label mapping to /kaggle/working/ddi_resnet_crf_label_mapping.json


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(type(tokenizer))
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>
Vocab size: 30522


In [10]:
def align_labels_with_offsets(offsets, entities, label2id):
    token_labels = [label2id["O"] for _ in offsets]
    eval_mask = []

    for idx, (start, end) in enumerate(offsets):
        is_special = (start == 0 and end == 0)
        eval_mask.append(0 if is_special else 1)

    occupied = [False for _ in offsets]
    for ent in sorted(entities, key=lambda x: (x["start"], x["end"])):
        first_token = True
        ent_start, ent_end, ent_type = ent["start"], ent["end"], ent["type"]
        b_label = label2id[f"B-{ent_type}"]
        i_label = label2id[f"I-{ent_type}"]

        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start == 0 and tok_end == 0:
                continue
            overlaps = tok_start < ent_end and tok_end > ent_start
            if overlaps and not occupied[i]:
                token_labels[i] = b_label if first_token else i_label
                occupied[i] = True
                first_token = False

    return token_labels, eval_mask


class DDINerDataset(Dataset):
    def __init__(self, examples, tokenizer, label2id, max_length=192):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(
            ex["text"],
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            add_special_tokens=True,
        )
        offsets = encoding.pop("offset_mapping")
        if not any(end > start for start, end in offsets):
            raise ValueError(f"No labelable tokens in sentence {ex['id']!r}.")
        labels, eval_mask = align_labels_with_offsets(offsets, ex["entities"], self.label2id)

        return {
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "eval_mask": torch.tensor(eval_mask, dtype=torch.long),
        }


def collate_batch(batch):
    max_len = max(item["input_ids"].size(0) for item in batch)
    pad_id = tokenizer.pad_token_id
    o_id = label2id["O"]

    out = defaultdict(list)
    for item in batch:
        length = item["input_ids"].size(0)
        pad_len = max_len - length
        out["input_ids"].append(torch.cat([item["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        out["attention_mask"].append(torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        out["labels"].append(torch.cat([item["labels"], torch.full((pad_len,), o_id, dtype=torch.long)]))
        out["eval_mask"].append(torch.cat([item["eval_mask"], torch.zeros(pad_len, dtype=torch.long)]))

    return {key: torch.stack(value) for key, value in out.items()}

train_ds = DDINerDataset(train_examples, tokenizer, label2id, MAX_LENGTH)
valid_ds = DDINerDataset(valid_examples, tokenizer, label2id, MAX_LENGTH)
test_ds = DDINerDataset(test_examples, tokenizer, label2id, MAX_LENGTH) if test_examples else None

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=NUM_WORKERS) if test_ds else None

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("Decoded sample:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0][:30]))
print("Label sample:", [id2label[i.item()] for i in batch["labels"][0][:30]])

{'input_ids': (8, 36), 'attention_mask': (8, 36), 'labels': (8, 36), 'eval_mask': (8, 36)}
Decoded sample: ['[CLS]', 'buprenorphine', 'is', 'metabolized', 'to', 'nor', '##bu', '##prenorphine', 'by', 'cytochrome', 'cyp', '3a', '##4', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Label sample: ['O', 'B-drug', 'O', 'O', 'O', 'B-drug_n', 'I-drug_n', 'I-drug_n', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


## Kiến trúc

Cell dưới là bản sao của `resnet_crf.py` để notebook chạy độc lập.
Các vị trí `eval_mask=1` được đóng gói thành chuỗi liên tục trước khi đưa vào CRF.


In [11]:
"""Length-preserving residual Conv1D + linear-chain CRF for flat BIO NER.

ResNetCRF accepts word/subword features; BertResNetCRF adds a Hugging Face
encoder. ``tag_mask`` selects positions to label (exclude special tokens or
select first subwords for word-level tagging). Decoding returns -100 at
unselected positions and omits right padding, preserving input alignment.
"""

from collections.abc import Sequence

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence


def _validate_mask(mask: torch.Tensor, shape: tuple[int, int]) -> torch.Tensor:
    if mask.shape != shape or len(shape) != 2 or min(shape) == 0:
        raise ValueError("Expected a nonempty [batch, length] mask.")
    if not ((mask == 0) | (mask == 1)).all():
        raise ValueError("Masks must contain only 0/1 values.")
    mask = mask.bool()
    if not mask[:, 0].all() or (mask[:, 1:] & ~mask[:, :-1]).any():
        raise ValueError("Each sequence must be nonempty and right padded.")
    return mask


class LinearChainCRF(nn.Module):
    """Mean sequence negative log-likelihood and Viterbi decoding.

    Transitions are indexed [previous_tag, next_tag]. Passing BIO label names
    enables hard BIO constraints in both the partition function and decoding.
    Padding labels may be -100; labels at valid positions must be tag IDs.
    """

    def __init__(self, num_tags: int, labels: Sequence[str] | None = None):
        super().__init__()
        if num_tags < 1:
            raise ValueError("num_tags must be positive.")
        self.num_tags = num_tags
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        start_allowed = torch.ones(num_tags, dtype=torch.bool)
        allowed = torch.ones(num_tags, num_tags, dtype=torch.bool)
        if labels is not None:
            if len(labels) != num_tags or len(set(labels)) != num_tags or "O" not in labels:
                raise ValueError("Provide unique BIO labels matching num_tags, including O.")
            for j, label in enumerate(labels):
                if label != "O" and (label[:2] not in ("B-", "I-") or not label[2:]):
                    raise ValueError(f"Not a BIO label: {label!r}")
                if label.startswith("I-"):
                    if f"B-{label[2:]}" not in labels:
                        raise ValueError(f"Missing B tag for {label!r}.")
                    start_allowed[j] = False
                    for i, previous in enumerate(labels):
                        allowed[i, j] = previous in (f"B-{label[2:]}", label)
        self.register_buffer("start_allowed", start_allowed)
        self.register_buffer("transition_allowed", allowed)
        for parameter in self.parameters():
            nn.init.uniform_(parameter, -0.1, 0.1)

    def _prepare(self, emissions, mask):
        if emissions.ndim != 3 or emissions.size(-1) != self.num_tags:
            raise ValueError("emissions must have shape [batch, length, num_tags].")
        mask = _validate_mask(mask, emissions.shape[:2])
        # Dynamic programming stays in float32 even inside mixed precision.
        emissions = emissions.float().masked_fill(~mask.unsqueeze(-1), 0.0)
        start = self.start_transitions.float().masked_fill(~self.start_allowed, -torch.inf)
        transitions = self.transitions.float().masked_fill(~self.transition_allowed, -torch.inf)
        return emissions, mask, start, transitions

    def forward(self, emissions, tags, mask):
        emissions, mask, start, transitions = self._prepare(emissions, mask)
        if tags.shape != mask.shape or tags.dtype != torch.long:
            raise ValueError("tags must be int64 with shape [batch, length].")
        if ((tags[mask] < 0) | (tags[mask] >= self.num_tags)).any():
            raise ValueError("Valid positions must contain tag IDs, not ignore indices.")
        tags = tags.masked_fill(~mask, 0)
        if not self.start_allowed[tags[:, 0]].all():
            raise ValueError("Gold BIO sequences cannot start with I tags.")
        if ((~self.transition_allowed[tags[:, :-1], tags[:, 1:]]) & mask[:, 1:]).any():
            raise ValueError("Gold sequence contains an invalid BIO transition.")

        batch = torch.arange(emissions.size(0), device=emissions.device)
        gold = start[tags[:, 0]] + emissions[batch, 0, tags[:, 0]]
        score = start + emissions[:, 0]
        for t in range(1, emissions.size(1)):
            step = transitions[tags[:, t - 1], tags[:, t]] + emissions[batch, t, tags[:, t]]
            gold = gold + torch.where(mask[:, t], step, 0.0)
            candidates = score.unsqueeze(2) + transitions + emissions[:, t].unsqueeze(1)
            next_score = torch.logsumexp(candidates, dim=1)
            score = torch.where(mask[:, t, None], next_score, score)
        last = mask.long().sum(1) - 1
        gold = gold + self.end_transitions.float()[tags[batch, last]]
        partition = torch.logsumexp(score + self.end_transitions.float(), dim=1)
        return (partition - gold).mean()

    @torch.no_grad()
    def decode(self, emissions, mask):
        emissions, mask, start, transitions = self._prepare(emissions, mask)
        score = start + emissions[:, 0]
        history = []
        for t in range(1, emissions.size(1)):
            candidates = score.unsqueeze(2) + transitions + emissions[:, t].unsqueeze(1)
            best_score, previous = candidates.max(dim=1)
            score = torch.where(mask[:, t, None], best_score, score)
            history.append(previous)
        last_tags = (score + self.end_transitions.float()).argmax(dim=1)
        paths = []
        for b, length in enumerate(mask.sum(1).tolist()):
            tag = last_tags[b].item()
            path = [tag]
            for previous in reversed(history[:length - 1]):
                tag = previous[b, tag].item()
                path.append(tag)
            paths.append(path[::-1])
        return paths


class ResidualConv1DBlock(nn.Module):
    """Two same-length convolutions; LayerNorm is applied per position.

    Masking after every convolution prevents padded activations from leaking
    back into valid tokens in the next layer. LayerNorm avoids statistics
    depending on padding length or the other examples in the batch.
    """

    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1,
                 dropout: float = 0.2):
        super().__init__()
        if kernel_size < 1 or kernel_size % 2 == 0 or dilation < 1:
            raise ValueError("Use a positive odd kernel size and positive dilation.")
        padding = dilation * (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding, dilation=dilation)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, features, mask):
        padding = ~mask.bool().unsqueeze(-1)
        residual = features.masked_fill(padding, 0.0)
        hidden = self.conv1(residual.transpose(1, 2)).transpose(1, 2)
        hidden = self.dropout(self.activation(self.norm1(hidden))).masked_fill(padding, 0.0)
        hidden = self.conv2(hidden.transpose(1, 2)).transpose(1, 2)
        hidden = self.dropout(self.norm2(hidden)).masked_fill(padding, 0.0)
        return self.activation(residual + hidden).masked_fill(padding, 0.0)


class ResNetCRF(nn.Module):
    """Reusable NER head for [batch, length, input_size] contextual features."""

    def __init__(self, input_size: int, num_labels: int, channels: int = 256,
                 num_blocks: int = 3, kernel_size: int = 3, dropout: float = 0.2,
                 dilations: Sequence[int] | None = None,
                 labels: Sequence[str] | None = None):
        super().__init__()
        if min(input_size, channels, num_blocks) < 1:
            raise ValueError("input_size, channels and num_blocks must be positive.")
        dilations = tuple(dilations) if dilations is not None else (1,) * num_blocks
        if len(dilations) != num_blocks:
            raise ValueError("Provide one dilation per residual block.")
        self.input_projection = nn.Linear(input_size, channels)
        self.input_norm = nn.LayerNorm(channels)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            ResidualConv1DBlock(channels, kernel_size, dilation, dropout)
            for dilation in dilations
        ])
        self.classifier = nn.Linear(channels, num_labels)
        self.crf = LinearChainCRF(num_labels, labels)

    def emissions(self, features, attention_mask):
        if features.ndim != 3:
            raise ValueError("features must have shape [batch, length, input_size].")
        mask = _validate_mask(attention_mask, features.shape[:2])
        # Clear padding before projection as well as after each residual layer.
        features = features.masked_fill(~mask.unsqueeze(-1), 0.0)
        hidden = self.dropout(self.input_norm(self.input_projection(features)))
        for block in self.blocks:
            hidden = block(hidden, mask)
        return self.classifier(self.dropout(hidden)).masked_fill(~mask.unsqueeze(-1), 0.0)

    @staticmethod
    def _pack(emissions, attention_mask, tag_mask=None, labels=None):
        mask = _validate_mask(attention_mask, emissions.shape[:2])
        selected = mask if tag_mask is None else tag_mask
        if selected.shape != mask.shape or not ((selected == 0) | (selected == 1)).all():
            raise ValueError("tag_mask must be a binary mask matching attention_mask.")
        selected = selected.bool()
        if (selected & ~mask).any() or not selected.any(dim=1).all():
            raise ValueError("Select at least one non-padding token in every sequence.")
        if labels is not None and (labels.shape != mask.shape or labels.dtype != torch.long):
            raise ValueError("labels must be int64 with shape [batch, length].")
        packed = pad_sequence([row[keep] for row, keep in zip(emissions, selected)], batch_first=True)
        lengths = selected.sum(1)
        packed_mask = torch.arange(packed.size(1), device=emissions.device)[None] < lengths[:, None]
        packed_labels = None
        if labels is not None:
            packed_labels = pad_sequence([row[keep] for row, keep in zip(labels, selected)],
                                         batch_first=True, padding_value=-100)
        return packed, packed_mask, packed_labels, selected

    def loss_from_emissions(self, emissions, labels, attention_mask, tag_mask=None):
        packed, mask, tags, _ = self._pack(emissions, attention_mask, tag_mask, labels)
        return self.crf(packed, tags, mask)

    @torch.no_grad()
    def decode_emissions(self, emissions, attention_mask, tag_mask=None):
        packed, mask, _, selected = self._pack(emissions, attention_mask, tag_mask)
        paths = self.crf.decode(packed, mask)
        aligned = []
        for b, path in enumerate(paths):
            row = [-100] * int(attention_mask[b].sum().item())
            for position, tag in zip(selected[b].nonzero(as_tuple=True)[0].tolist(), path):
                row[position] = tag
            aligned.append(row)
        return aligned

    def forward(self, features, attention_mask, labels=None, tag_mask=None):
        emissions = self.emissions(features, attention_mask)
        if labels is None:
            return emissions
        return self.loss_from_emissions(emissions, labels, attention_mask, tag_mask), emissions

    @torch.no_grad()
    def decode(self, features, attention_mask, tag_mask=None):
        return self.decode_emissions(self.emissions(features, attention_mask), attention_mask, tag_mask)


class BertResNetCRF(nn.Module):
    """Pretrained text encoder -> residual Conv1D head -> CRF.

    An encoder can be injected for offline testing. It must expose
    config.hidden_size and return an object with last_hidden_state.
    """

    def __init__(self, model_name: str | None, num_labels: int, *, encoder=None, **head_kwargs):
        super().__init__()
        if encoder is None:
            if model_name is None:
                raise ValueError("Provide model_name or encoder.")
            from transformers import AutoModel
            encoder = AutoModel.from_pretrained(model_name)
        self.bert = encoder
        self.head = ResNetCRF(self.bert.config.hidden_size, num_labels, **head_kwargs)

    def emissions(self, input_ids, attention_mask):
        _validate_mask(attention_mask, input_ids.shape)
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.head.emissions(outputs.last_hidden_state, attention_mask)

    def forward(self, input_ids, attention_mask, labels=None, tag_mask=None):
        emissions = self.emissions(input_ids, attention_mask)
        if labels is None:
            return emissions
        return self.head.loss_from_emissions(emissions, labels, attention_mask, tag_mask), emissions

    @torch.no_grad()
    def decode(self, input_ids, attention_mask, tag_mask=None):
        return self.head.decode_emissions(self.emissions(input_ids, attention_mask), attention_mask, tag_mask)


In [12]:
model = BertResNetCRF(
    MODEL_NAME, num_labels=len(labels), labels=labels, **MODEL_CONFIG,
).to(device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("ResNet/CRF parameters:", sum(p.numel() for p in model.head.parameters()))


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Trainable parameters: 110866284
ResNet/CRF parameters: 1384044


In [13]:
def get_entities_from_bio(tags):
    entities = set()
    ent_type = None
    start = None

    for i, tag in enumerate(tags):
        if tag == "O" or tag is None:
            if ent_type is not None:
                entities.add((ent_type, start, i - 1))
                ent_type, start = None, None
            continue

        prefix, typ = tag.split("-", 1) if "-" in tag else ("O", None)
        if prefix == "B" or ent_type != typ:
            if ent_type is not None:
                entities.add((ent_type, start, i - 1))
            ent_type, start = typ, i
        elif prefix == "I":
            continue
        else:
            if ent_type is not None:
                entities.add((ent_type, start, i - 1))
                ent_type, start = None, None

    if ent_type is not None:
        entities.add((ent_type, start, len(tags) - 1))
    return entities


def evaluate(model, dataloader, id2label, desc="Evaluating"):
    model.eval()
    total_loss = 0.0
    n_examples = 0
    token_correct = 0
    token_total = 0
    gold_total = 0
    pred_total = 0
    correct_total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_tensor = batch["labels"].to(device)
            eval_mask = batch["eval_mask"].to(device)

            loss, emissions = model(input_ids, attention_mask, labels_tensor, tag_mask=eval_mask)
            paths = model.head.decode_emissions(emissions, attention_mask, tag_mask=eval_mask)

            total_loss += loss.item() * input_ids.size(0)
            n_examples += input_ids.size(0)

            for b, path in enumerate(paths):
                gold_tags = []
                pred_tags = []
                valid_positions = eval_mask[b].bool().cpu().tolist()
                gold_ids = labels_tensor[b].detach().cpu().tolist()

                for pos, keep in enumerate(valid_positions[:len(path)]):
                    if not keep:
                        continue
                    gold = id2label[gold_ids[pos]]
                    pred = id2label[path[pos]]
                    gold_tags.append(gold)
                    pred_tags.append(pred)
                    token_correct += int(gold == pred)
                    token_total += 1

                gold_entities = get_entities_from_bio(gold_tags)
                pred_entities = get_entities_from_bio(pred_tags)
                gold_total += len(gold_entities)
                pred_total += len(pred_entities)
                correct_total += len(gold_entities & pred_entities)

    precision = correct_total / pred_total if pred_total else 0.0
    recall = correct_total / gold_total if gold_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    token_acc = token_correct / token_total if token_total else 0.0

    return {
        "loss": total_loss / max(n_examples, 1),
        "token_acc": token_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "gold_entities": gold_total,
        "pred_entities": pred_total,
        "correct_entities": correct_total,
    }

In [14]:
bert_params = list(model.bert.parameters())
head_params = list(model.head.parameters())

optimizer = AdamW(
    [
        {"params": bert_params, "lr": LR_BERT, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
    ]
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
print("Total steps:", total_steps, "| Warmup steps:", warmup_steps)

Total steps: 7770 | Warmup steps: 777


In [15]:
best_valid_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    examples_seen = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        gold = batch["labels"].to(device)
        tag_mask = batch["eval_mask"].to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss, _ = model(input_ids, attention_mask, gold, tag_mask=tag_mask)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= previous_scale:
            scheduler.step()  # Do not advance if AMP skipped an overflowing update.
        train_loss += loss.item() * input_ids.size(0)
        examples_seen += input_ids.size(0)
        progress.set_postfix(loss=f"{train_loss / examples_seen:.4f}")

    valid_metrics = evaluate(model, valid_loader, id2label, desc=f"Valid epoch {epoch}")
    row = {"epoch": epoch, "train_loss": train_loss / examples_seen,
           **{f"valid_{key}": value for key, value in valid_metrics.items()}}
    history.append(row)
    print(row)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "biomedbert_resnet_crf_history.csv", index=False)
    if valid_metrics["f1"] > best_valid_f1:
        best_valid_f1 = valid_metrics["f1"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "model_config": MODEL_CONFIG,
            "label2id": label2id,
            "id2label": id2label,
            "max_length": MAX_LENGTH,
            "seed": SEED,
            "epoch": epoch,
            "valid_metrics": valid_metrics,
            "architecture": "BiomedBERT + ResNet1D + CRF",
            "label_unit": "subword",
            "split_unit": "sentence",
        }, BEST_MODEL_PATH)
        tokenizer.save_pretrained(OUTPUT_DIR / "biomedbert_resnet_crf_tokenizer")
        print("Saved best validation checkpoint:", BEST_MODEL_PATH)

history_df = pd.DataFrame(history)
history_df


Epoch 1/10:   0%|          | 0/777 [00:00<?, ?it/s]

Valid epoch 1:   0%|          | 0/44 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 3.6497445217371944, 'valid_loss': 1.3083870321080304, 'valid_token_acc': 0.968725783239234, 'valid_precision': 0.9250909090909091, 'valid_recall': 0.8712328767123287, 'valid_f1': 0.8973544973544973, 'valid_gold_entities': 1460, 'valid_pred_entities': 1375, 'valid_correct_entities': 1272}
Saved best validation checkpoint: /kaggle/working/biomedbert_resnet_crf_ddi_best.pt


Epoch 2/10:   0%|          | 0/777 [00:00<?, ?it/s]

Valid epoch 2:   0%|          | 0/44 [00:00<?, ?it/s]

...

{'epoch': 9, 'train_loss': 0.09070677515104797, 'valid_loss': 1.7258260765585347, 'valid_token_acc': 0.9851101516661132, 'valid_precision': 0.9454545454545454, 'valid_recall': 0.9616438356164384, 'valid_f1': 0.9534804753820034, 'valid_gold_entities': 1460, 'valid_pred_entities': 1485, 'valid_correct_entities': 1404}
Saved best validation checkpoint: /kaggle/working/biomedbert_resnet_crf_ddi_best.pt


Epoch 10/10:   0%|          | 0/777 [00:00<?, ?it/s]

Valid epoch 10:   0%|          | 0/44 [00:00<?, ?it/s]

{'epoch': 10, 'train_loss': 0.07693841862084037, 'valid_loss': 1.707001851779827, 'valid_token_acc': 0.9849994464740396, 'valid_precision': 0.9434724091520862, 'valid_recall': 0.9602739726027397, 'valid_f1': 0.9517990495587237, 'valid_gold_entities': 1460, 'valid_pred_entities': 1486, 'valid_correct_entities': 1402}


,epoch,train_loss,valid_loss,valid_token_acc,valid_precision,valid_recall,valid_f1,valid_gold_entities,valid_pred_entities,valid_correct_entities
0,1,3.649745,1.308387,0.968726,0.925091,0.871233,0.897354,1460,1375,1272
1,2,1.002120,1.203039,0.979686,0.920241,0.940411,0.930217,1460,1492,1373
2,3,0.663302,0.935085,0.983560,0.941980,0.945205,0.943590,1460,1465,1380
3,4,0.469716,1.337571,0.982564,0.923689,0.953425,0.938322,1460,1507,1392
4,5,0.341107,1.633254,0.983284,0.933422,0.960274,0.946658,1460,1502,1402
5,6,0.248619,1.445656,0.983173,0.939230,0.952740,0.945937,1460,1481,1391
6,7,0.173631,1.478466,0.984833,0.937751,0.959589,0.948544,1460,1494,1401
7,8,0.134063,1.659972,0.984723,0.936412,0.958219,0.947190,1460,1494,1399
8,9,0.090707,1.725826,0.985110,0.945455,0.961644,0.953480,1460,1485,1404
9,10,0.076938,1.707002,0.984999,0.943472,0.960274,0.951799,1460,1486,1402


In [16]:
# Load best checkpoint before final evaluation.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
print("Best validation metrics:", checkpoint["valid_metrics"])

if test_loader is not None and len(test_ds) > 0:
    test_metrics = evaluate(model, test_loader, id2label, desc="Test")
    print("Test metrics:")
    print(json.dumps(test_metrics, indent=2))
else:
    print("Khong co test_loader. Kiem tra lai TEST_DIR neu can danh gia test.")

Best validation metrics: {'loss': 1.7258260765585347, 'token_acc': 0.9851101516661132, 'precision': 0.9454545454545454, 'recall': 0.9616438356164384, 'f1': 0.9534804753820034, 'gold_entities': 1460, 'pred_entities': 1485, 'correct_entities': 1404}


Test:   0%|          | 0/42 [00:00<?, ?it/s]

Test metrics:
{
  "loss": 3.357442007746015,
  "token_acc": 0.9692148641663316,
  "precision": 0.7493224932249323,
  "recall": 0.8061224489795918,
  "f1": 0.7766853932584269,
  "gold_entities": 686,
  "pred_entities": 738,
  "correct_entities": 553
}


In [17]:
# Evaluate the best model separately on DrugBank and MedLine test subsets.
def load_ddi_examples_from_files(xml_files, desc="Reading XML files"):
    examples = []
    skipped = []
    for xml_file in tqdm(sorted(xml_files), desc=desc):
        try:
            examples.extend(read_xml_file(xml_file))
        except Exception as exc:
            skipped.append((str(xml_file), str(exc)))
    return examples, skipped


def find_test_xml_files_by_domain(domain_name):
    domain_key = domain_name.lower()
    candidate_dirs = [
        TEST_DIR / domain_name,
        TEST_DIR / domain_name.lower(),
        TEST_DIR / domain_name.upper(),
        TEST_ROOT / "Test for DrugNER task" / domain_name,
        TEST_ROOT / "Test for DrugNER task" / domain_name.lower(),
        TEST_ROOT / domain_name,
        TEST_ROOT / domain_name.lower(),
    ]

    for candidate in candidate_dirs:
        if candidate.exists():
            xml_files = sorted(candidate.rglob("*.xml"))
            if xml_files:
                return xml_files

    # Fallback: use path/file names containing the domain keyword.
    xml_files = []
    for xml_file in TEST_DIR.rglob("*.xml"):
        path_text = str(xml_file).lower()
        if domain_key in path_text:
            xml_files.append(xml_file)
    return sorted(xml_files)


def evaluate_test_domain(domain_name):
    xml_files = find_test_xml_files_by_domain(domain_name)
    print(f"{domain_name} XML files:", len(xml_files))

    if not xml_files:
        print(f"Khong tim thay XML cho {domain_name}. Hay kiem tra cau truc folder trong TEST_DIR:", TEST_DIR)
        return None

    examples, skipped = load_ddi_examples_from_files(xml_files, desc=f"Reading {domain_name}")
    print(f"{domain_name} sentences:", len(examples), "| skipped XML:", len(skipped))

    dataset = DDINerDataset(examples, tokenizer, label2id, MAX_LENGTH)
    loader = DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_batch,
        num_workers=NUM_WORKERS,
    )
    metrics = evaluate(model, loader, id2label, desc=f"Test {domain_name}")
    metrics = {"subset": domain_name, "sentences": len(examples), "xml_files": len(xml_files), **metrics}
    print(f"{domain_name} metrics:")
    print(json.dumps(metrics, indent=2))
    return metrics

# Make sure the best checkpoint is loaded before subset evaluation.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

subset_results = []
for subset_name in ["DrugBank", "MedLine"]:
    result = evaluate_test_domain(subset_name)
    if result is not None:
        subset_results.append(result)

subset_results_df = pd.DataFrame(subset_results)
subset_results_df

DrugBank XML files: 54


Reading DrugBank:   0%|          | 0/54 [00:00<?, ?it/s]

DrugBank sentences: 145 | skipped XML: 0


Test DrugBank:   0%|          | 0/10 [00:00<?, ?it/s]

DrugBank metrics:
{
  "subset": "DrugBank",
  "sentences": 145,
  "xml_files": 54,
  "loss": 3.0170226368410833,
  "token_acc": 0.9732044198895028,
  "precision": 0.8888888888888888,
  "recall": 0.9210526315789473,
  "f1": 0.9046849757673667,
  "gold_entities": 304,
  "pred_entities": 315,
  "correct_entities": 280
}
MedLine XML files: 58


Reading MedLine:   0%|          | 0/58 [00:00<?, ?it/s]

MedLine sentences: 520 | skipped XML: 0


Test MedLine:   0%|          | 0/33 [00:00<?, ?it/s]

MedLine metrics:
{
  "subset": "MedLine",
  "sentences": 520,
  "xml_files": 58,
  "loss": 3.452365973362556,
  "token_acc": 0.9681676455659488,
  "precision": 0.6453900709219859,
  "recall": 0.7146596858638743,
  "f1": 0.6782608695652174,
  "gold_entities": 382,
  "pred_entities": 423,
  "correct_entities": 273
}


,subset,sentences,xml_files,loss,token_acc,precision,recall,f1,gold_entities,pred_entities,correct_entities
0,DrugBank,145,54,3.017023,0.973204,0.888889,0.921053,0.904685,304,315,280
1,MedLine,520,58,3.452366,0.968168,0.645390,0.714660,0.678261,382,423,273


In [18]:
def predict_entities(text, model, tokenizer, id2label, max_length=192):
    model.eval()
    if not text.strip():
        return []
    model_device = next(model.parameters()).device
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        return_tensors="pt",
    )
    offsets = encoding.pop("offset_mapping")[0].tolist()
    encoding = {k: v.to(model_device) for k, v in encoding.items()}

    tag_mask = torch.tensor([[end > start for start, end in offsets]],
                            dtype=torch.bool, device=model_device)
    if not tag_mask.any():
        return []
    with torch.no_grad():
        path = model.decode(encoding["input_ids"], encoding["attention_mask"], tag_mask=tag_mask)[0]

    token_tags = []
    for tag_id, (start, end) in zip(path, offsets):
        if start == 0 and end == 0:
            continue
        token_tags.append((start, end, id2label[tag_id]))

    spans = []
    current = None
    for start, end, tag in token_tags:
        if tag == "O":
            if current:
                spans.append(current)
                current = None
            continue
        prefix, typ = tag.split("-", 1)
        if prefix == "B" or current is None or current["type"] != typ:
            if current:
                spans.append(current)
            current = {"type": typ, "start": start, "end": end}
        else:
            current["end"] = end
    if current:
        spans.append(current)

    for span in spans:
        span["text"] = text[span["start"]:span["end"]]
    return spans

sample_text = "Aspirin may increase the anticoagulant activities of Warfarin."
predict_entities(sample_text, model, tokenizer, id2label, MAX_LENGTH)

[{'type': 'brand', 'start': 0, 'end': 7, 'text': 'Aspirin'},
 {'type': 'drug', 'start': 53, 'end': 61, 'text': 'Warfarin'}]

## Tài liệu tham khảo

- Qiu et al. (2018), [Residual Dilated CNN–CRF cho clinical NER](https://arxiv.org/abs/1808.08669).
- Che et al. (2020), [TCN–CRF cho biomedical NER](https://doi.org/10.3934/mbe.2020200).

Bản này dùng LayerNorm theo vị trí và convolution không causal, dilation mặc định 1.
Chỉ lựa chọn cấu hình trên validation; không dùng test để chọn epoch/tham số.
